## General imports

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib as mp
import sklearn as sk
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, kruskal
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
import numpy as np

## Load Data

In [ ]:
dataset_train_ori = pd.read_csv("database_smartphone/train_data.csv")
dataset_test_ori = pd.read_csv("database_smartphone/test_data.csv")
data_test = dataset_test_ori.copy()
dataset = dataset_train_ori.copy()
dataset.head()
dataset.info()
dataset.shape

# Data cleaning

## Missing values

In [ ]:
# visualize null values
plt.figure(figsize=(10, 6))
sns.heatmap(dataset.isnull(), yticklabels=False, cmap='viridis', cbar=False)
dataset.isnull().sum()
data_test.isnull().sum()

In [ ]:
features_with_nan = dataset.columns[dataset.isna().any()]

missing_counts = dataset[features_with_nan].isna().sum()



print(missing_counts)

In [ ]:
print("Before cleaning:")
print(dataset['label'].value_counts(normalize=True))

dataset_clean = dataset.dropna()

print("After cleaning:")
print(dataset_clean['label'].value_counts(normalize=True))

print("Before cleaning:")
print(data_test['label'].value_counts(normalize=True))

data_test_clean = data_test.dropna()

print("After cleaning:")
print(data_test_clean['label'].value_counts(normalize=True))


In [ ]:
dataset_clean = dataset.dropna()
dataset_clean.shape

## Drop Duplicates

In [ ]:
print(dataset_clean.duplicated().any())
if dataset_clean.duplicated().any():
    dataset_clean.drop_duplicates()

print(data_test_clean.duplicated().any())
if data_test_clean.duplicated().any():
    data_test_clean.drop_duplicates()

# Data Visualization

In [ ]:
#setting figure size for future visualizations
sns.set(rc={'figure.figsize':(10,6)})
sns.set_style('white')
dataset_clean.describe()

In [ ]:


pvals = []
features = []

for feature in dataset_clean.columns.drop('label'):
    groups = [
        dataset_clean[dataset_clean['label'] == c][feature].dropna()
        for c in dataset_clean['label'].unique()
    ]
    stat, p = f_oneway(*groups)
    pvals.append(p)
    features.append(feature)

# correction FDR
reject, pvals_corr, _, _ = multipletests(
    pvals, alpha=0.05, method='fdr_bh'
)

anova_selected = [f for f, r in zip(features, reject) if r]
print(f"Features selected by ANOVA : {anova_selected}")

dataset_selected_features = dataset_clean[anova_selected + ["label"]]
print(dataset_selected_features.shape)
data_test_selected_features = data_test[anova_selected + ["label"]]
print(data_test_selected_features.shape)


In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(dataset_selected_features.corr(), annot=False, cmap='coolwarm')
plt.title("Feature Correlation Matrix")
plt.show()


In [ ]:
# Calculate correlation of each feature with the target variable y
corr_with_target = dataset_selected_features.corrwith(dataset_selected_features["label"])
print("Correlation of features with the target variable:")
print(corr_with_target)

# Set correlation threshold for feature removal
threshold = 0.8

dataset_selected_features_copy = dataset_selected_features.copy()
dataset_selected_features_copy.drop(columns = ["label"], inplace = True)
# Compute absolute correlation matrix
corr_matrix = dataset_selected_features_copy.corr().abs()

# Find pairs of features with correlation above threshold (upper triangle only)
correlated_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > threshold:
            correlated_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

print(f"Feature pairs with correlation greater than {threshold}:")
for pair in correlated_pairs:
    print(pair)

# Decide which feature to remove based on lower correlation with target
features_to_remove = set()
for feat1, feat2, corr_value in correlated_pairs:
    if abs(corr_with_target[feat1]) > abs(corr_with_target[feat2]):
        features_to_remove.add(feat2)
    else:
        features_to_remove.add(feat1)

print("Features to remove due to high correlation:")
print(features_to_remove)

# Create a new DataFrame without the removed features
df_reduced = dataset_selected_features.drop(columns=features_to_remove)
df_test_reduced = data_test_selected_features.drop(columns = features_to_remove)
print("Remaining features:")
print(df_reduced.columns.tolist())
print(df_reduced.shape)


In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(df_reduced.corr(), annot=False, cmap='coolwarm')
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df_reduced.drop(columns='label')
y = df_reduced['label']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(20).plot(kind='barh')
plt.title('Top 20 Feature Importances')
plt.show()


In [ ]:
class_names = {
    1: "walking",
    3: "shuffling",
    4: "stairs ascending",
    5: "stairs descending",
    6: "standing",
    7: "sitting",
    8: "lying"
}

df_plot = df_reduced.copy()

# Mapping label → activity
df_plot['activity'] = df_plot['label'].map(class_names)

features = df_plot.drop(columns=['label', 'activity']).columns.tolist()

plt.figure(figsize=(15, 3 * len(features)))

for i, feature in enumerate(features):
    plt.subplot(len(features), 1, i + 1)
    sns.boxplot(
        x='activity',
        y=feature,
        data=df_plot,
        showfliers=False  
    )
    plt.title(f'Distribution of {feature} by Activity')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
df_reduced.to_csv("database_smartphone/train_data_reduced.csv")
df_test_reduced.to_csv("database_smartphone/test_data_reduced.csv")

